<a href="https://colab.research.google.com/github/Ajwad07/ML_projects_/blob/main/SAGAN_paper_but_with_cifar10_imc_param_sweep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt

# Parameters
IMG_SIZE = 32
N_CHANNELS = 3
N_CLASSES = 10
LABEL_EMB_DIM = 16
LATENT_DIM = 128
NOISE_DIM = 128
BATCH_SIZE = 128
EPOCHS_AE = 20
EPOCHS_GAN = 50
LAMBDA_GP = 10
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
])

trainset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
testset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
test_loader = DataLoader(testset, batch_size=16, shuffle=True)


In [ ]:
class ConditionalEncoder(nn.Module):
    def __init__(self, latent_dim, n_classes, label_emb_dim):
        super().__init__()
        self.label_emb = nn.Embedding(n_classes, label_emb_dim)
        self.conv = nn.Sequential(
            nn.Conv2d(N_CHANNELS + label_emb_dim, 64, 3, 2, 1), nn.ReLU(),
            nn.Conv2d(64, 128, 3, 2, 1), nn.ReLU(),
            nn.Conv2d(128, 256, 3, 2, 1), nn.ReLU()
        )
        self.flatten = nn.Flatten()
        self.fc = nn.Sequential(
            nn.Linear(256 * (IMG_SIZE // 8) * (IMG_SIZE // 8), 256), nn.ReLU(),
            nn.Linear(256, latent_dim)
        )

    def forward(self, x, labels):
        label_emb = self.label_emb(labels)
        label_map = label_emb.unsqueeze(2).unsqueeze(3).expand(-1, -1, IMG_SIZE, IMG_SIZE)
        x = torch.cat([x, label_map], dim=1)
        x = self.conv(x)
        x = self.flatten(x)
        z = self.fc(x)
        return z

class ConditionalDecoder(nn.Module):
    def __init__(self, latent_dim, n_classes, label_emb_dim):
        super().__init__()
        self.label_emb = nn.Embedding(n_classes, label_emb_dim)
        self.fc = nn.Sequential(
            nn.Linear(latent_dim + label_emb_dim, 256), nn.ReLU(),
            nn.Linear(256, 256 * (IMG_SIZE // 8) * (IMG_SIZE // 8)), nn.ReLU()
        )
        self.unflatten = nn.Unflatten(1, (256, IMG_SIZE // 8, IMG_SIZE // 8))
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.ReLU(),
            nn.ConvTranspose2d(64, N_CHANNELS, 4, 2, 1), nn.Sigmoid()
        )

    def forward(self, z, labels):
        label_emb = self.label_emb(labels)
        z_cond = torch.cat([z, label_emb], dim=1)
        x = self.fc(z_cond)
        x = self.unflatten(x)
        x = self.deconv(x)
        return x


In [ ]:
class LatentGenerator(nn.Module):
    def __init__(self, noise_dim, latent_dim, n_classes, label_emb_dim):
        super().__init__()
        self.label_emb = nn.Embedding(n_classes, label_emb_dim)
        self.fc = nn.Sequential(
            nn.Linear(noise_dim + label_emb_dim, 256), nn.ReLU(),
            nn.Linear(256, latent_dim)
        )

    def forward(self, noise, labels):
        label_emb = self.label_emb(labels)
        x = torch.cat([noise, label_emb], dim=1)
        return self.fc(x)

class ConditionalDiscriminator(nn.Module):
    def __init__(self, n_classes, label_emb_dim):
        super().__init__()
        self.label_emb = nn.Embedding(n_classes, label_emb_dim)
        self.conv = nn.Sequential(
            nn.Conv2d(N_CHANNELS + label_emb_dim, 64, 3, 2, 1), nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, 3, 2, 1), nn.LeakyReLU(0.2),
            nn.Conv2d(128, 256, 3, 2, 1), nn.LeakyReLU(0.2)
        )
        self.flatten = nn.Flatten()
        self.fc = nn.Sequential(
            nn.Linear(256 * (IMG_SIZE // 8) * (IMG_SIZE // 8), 64), nn.LeakyReLU(0.2),
            nn.Linear(64, 1), nn.Sigmoid()
        )

    def forward(self, x, labels):
        label_emb = self.label_emb(labels)
        label_map = label_emb.unsqueeze(2).unsqueeze(3).expand(-1, -1, IMG_SIZE, IMG_SIZE)
        x = torch.cat([x, label_map], dim=1)
        x = self.conv(x)
        x = self.flatten(x)
        return self.fc(x)


In [ ]:
encoder = ConditionalEncoder(LATENT_DIM, N_CLASSES, LABEL_EMB_DIM).to(DEVICE)
decoder = ConditionalDecoder(LATENT_DIM, N_CLASSES, LABEL_EMB_DIM).to(DEVICE)

optimizer_ae = optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=1e-3)
criterion_ae = nn.MSELoss()

for epoch in range(EPOCHS_AE):
    encoder.train()
    decoder.train()
    total_loss = 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        z = encoder(imgs, labels)
        recons = decoder(z, labels)
        loss = criterion_ae(recons, imgs)
        optimizer_ae.zero_grad()
        loss.backward()
        optimizer_ae.step()
        total_loss += loss.item() * imgs.size(0)
    avg_loss = total_loss / len(train_loader.dataset)
    print(f"[AE] Epoch {epoch+1}/{EPOCHS_AE} - Loss: {avg_loss:.4f}")


[AE] Epoch 1/20 - Loss: 0.0294
[AE] Epoch 2/20 - Loss: 0.0174
[AE] Epoch 3/20 - Loss: 0.0138
[AE] Epoch 4/20 - Loss: 0.0124
[AE] Epoch 5/20 - Loss: 0.0115
[AE] Epoch 6/20 - Loss: 0.0108
[AE] Epoch 7/20 - Loss: 0.0103
[AE] Epoch 8/20 - Loss: 0.0097
[AE] Epoch 9/20 - Loss: 0.0094
[AE] Epoch 10/20 - Loss: 0.0089
[AE] Epoch 11/20 - Loss: 0.0086
[AE] Epoch 12/20 - Loss: 0.0084
[AE] Epoch 13/20 - Loss: 0.0082
[AE] Epoch 14/20 - Loss: 0.0081
[AE] Epoch 15/20 - Loss: 0.0079
[AE] Epoch 16/20 - Loss: 0.0078
[AE] Epoch 17/20 - Loss: 0.0077
[AE] Epoch 18/20 - Loss: 0.0075
[AE] Epoch 19/20 - Loss: 0.0074
[AE] Epoch 20/20 - Loss: 0.0073


In [ ]:
def compute_gradient_penalty(D, real_imgs, fake_imgs, real_labels, device, lambda_gp=LAMBDA_GP):
    alpha = torch.rand(real_imgs.size(0), 1, 1, 1, device=device)
    interpolates = (alpha * real_imgs + (1 - alpha) * fake_imgs).requires_grad_(True)
    d_interpolates = D(interpolates, real_labels)
    fake = torch.ones(d_interpolates.size(), device=device, requires_grad=False)
    gradients = torch.autograd.grad(
        outputs=d_interpolates,
        inputs=interpolates,
        grad_outputs=fake,
        create_graph=True,
        retain_graph=True,
        only_inputs=True
    )[0]
    gradients = gradients.view(gradients.size(0), -1)
    gradient_norm = gradients.norm(2, dim=1)
    gp = lambda_gp * ((gradient_norm - 1) ** 2).mean()
    return gp


In [ ]:
G = LatentGenerator(NOISE_DIM, LATENT_DIM, N_CLASSES, LABEL_EMB_DIM).to(DEVICE)
D = ConditionalDiscriminator(N_CLASSES, LABEL_EMB_DIM).to(DEVICE)
decoder.eval()  # Decoder is fixed!

optimizer_G = optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
optimizer_D = optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))
criterion_bce = nn.BCELoss()

for epoch in range(EPOCHS_GAN):
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        batch_size = imgs.size(0)
        # Generate fake images
        z = torch.randn(batch_size, NOISE_DIM, device=DEVICE)
        fake_labels = torch.randint(0, N_CLASSES, (batch_size,), device=DEVICE)
        z_fake = G(z, fake_labels)
        with torch.no_grad():
            x_fake = decoder(z_fake, fake_labels)
        # Wrong labels (shuffled)
        wrong_labels = (labels + torch.randint(1, N_CLASSES, (batch_size,), device=DEVICE)) % N_CLASSES

        # Discriminator step
        real = torch.ones(batch_size, 1, device=DEVICE)
        fake = torch.zeros(batch_size, 1, device=DEVICE)
        # Real image, correct label
        pred_real = D(imgs, labels)
        loss_real = criterion_bce(pred_real, real)
        # Fake image, correct label
        pred_fake = D(x_fake, fake_labels)
        loss_fake = criterion_bce(pred_fake, fake)
        # Real image, wrong label
        pred_wrong = D(imgs, wrong_labels)
        loss_wrong = criterion_bce(pred_wrong, fake)
        # Gradient penalty
        gp = compute_gradient_penalty(D, imgs.data, x_fake.data, labels, DEVICE)
        # Total D loss
        d_loss = loss_real + loss_fake + loss_wrong + gp
        optimizer_D.zero_grad()
        d_loss.backward()
        optimizer_D.step()

        # Generator step
        z = torch.randn(batch_size, NOISE_DIM, device=DEVICE)
        gen_labels = torch.randint(0, N_CLASSES, (batch_size,), device=DEVICE)
        z_gen = G(z, gen_labels)
        x_gen = decoder(z_gen, gen_labels)
        pred = D(x_gen, gen_labels)
        g_loss = criterion_bce(pred, real)
        optimizer_G.zero_grad()
        g_loss.backward()
        optimizer_G.step()
    print(f"[GAN] Epoch {epoch+1}/{EPOCHS_GAN} | D_loss: {d_loss.item():.4f} | G_loss: {g_loss.item():.4f}")


In [ ]:
G.eval()
decoder.eval()
n_samples = 10
noise = torch.randn(n_samples, NOISE_DIM, device=DEVICE)
sample_labels = torch.arange(n_samples, device=DEVICE) % N_CLASSES
z = G(noise, sample_labels)
with torch.no_grad():
    fake_imgs = decoder(z, sample_labels).cpu()
plt.figure(figsize=(14,2))
for i in range(n_samples):
    plt.subplot(1, n_samples, i+1)
    plt.imshow(np.clip(fake_imgs[i].permute(1,2,0).numpy(), 0, 1))
    plt.title(f"{sample_labels[i].item()}")
    plt.axis('off')
plt.suptitle("Class-conditional synthetic samples")
plt.show()


# Task
Perform a hyperparameter sweep for a GAN model, tuning the generator learning rate, discriminator learning rate, and LAMBDA_GP to find the combination that results in the most stable training.

## Identify hyperparameters for sweep

### Subtask:
Define the hyperparameters that will be tuned: generator learning rate, discriminator learning rate, and LAMBDA_GP.


**Reasoning**:
The subtask is to identify the hyperparameters that will be tuned. These are explicitly stated in the instructions: generator learning rate, discriminator learning rate, and LAMBDA_GP. I will list these hyperparameters.



In [ ]:
# Hyperparameters to tune
hyperparameters_to_tune = [
    "generator learning rate",
    "discriminator learning rate",
    "LAMBDA_GP"
]

print("Hyperparameters to be tuned:")
for hp in hyperparameters_to_tune:
    print(f"- {hp}")

Hyperparameters to be tuned:
- generator learning rate
- discriminator learning rate
- LAMBDA_GP


## Define value ranges for sweep

### Subtask:
Specify the ranges of values to explore for each hyperparameter.


**Reasoning**:
Define the ranges for the generator learning rate, discriminator learning rate, and LAMBDA_GP for the hyperparameter sweep.



In [ ]:
# Define ranges for hyperparameters
lr_G_values = [1e-4, 2e-4, 4e-4]
lr_D_values = [1e-4, 2e-4, 4e-4]
lambda_gp_values = [5, 10, 15]

print("Hyperparameter ranges defined:")
print(f"Generator Learning Rates: {lr_G_values}")
print(f"Discriminator Learning Rates: {lr_D_values}")
print(f"LAMBDA_GP values: {lambda_gp_values}")

Hyperparameter ranges defined:
Generator Learning Rates: [0.0001, 0.0002, 0.0004]
Discriminator Learning Rates: [0.0001, 0.0002, 0.0004]
LAMBDA_GP values: [5, 10, 15]


## Implement hyperparameter sweep

### Subtask:
Implement hyperparameter sweep by iterating through different hyperparameter combinations, updating model optimizers and LAMBDA_GP accordingly, and running the GAN training loop for a reduced number of epochs for each combination.


**Reasoning**:
Implement the hyperparameter sweep by iterating through combinations and training the GAN for a reduced number of epochs while storing the losses.



In [ ]:
import itertools

# Reduced number of epochs for the sweep
SWEEP_EPOCHS = 10

# Store results
sweep_results = []

# Define ranges for hyperparameters (already defined, re-listing for clarity)
lr_G_values = [1e-4, 2e-4, 4e-4]
lr_D_values = [1e-4, 2e-4, 4e-4]
lambda_gp_values = [5, 10, 15]

# Iterate through all combinations
for lr_G, lr_D, lambda_gp in itertools.product(lr_G_values, lr_D_values, lambda_gp_values):
    print(f"Testing combination: lr_G={lr_G}, lr_D={lr_D}, LAMBDA_GP={lambda_gp}")

    # Re-initialize models and optimizers
    G = LatentGenerator(NOISE_DIM, LATENT_DIM, N_CLASSES, LABEL_EMB_DIM).to(DEVICE)
    D = ConditionalDiscriminator(N_CLASSES, LABEL_EMB_DIM).to(DEVICE)
    decoder.eval() # Decoder is fixed!

    optimizer_G = optim.Adam(G.parameters(), lr=lr_G, betas=(0.5, 0.999))
    optimizer_D = optim.Adam(D.parameters(), lr=lr_D, betas=(0.5, 0.999))

    # Store losses for this combination
    d_losses = []
    g_losses = []

    # Training loop for reduced epochs
    for epoch in range(SWEEP_EPOCHS):
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            batch_size = imgs.size(0)

            # Generate fake images
            z = torch.randn(batch_size, NOISE_DIM, device=DEVICE)
            fake_labels = torch.randint(0, N_CLASSES, (batch_size,), device=DEVICE)
            z_fake = G(z, fake_labels)
            with torch.no_grad():
                x_fake = decoder(z_fake, fake_labels)

            # Wrong labels (shuffled)
            wrong_labels = (labels + torch.randint(1, N_CLASSES, (batch_size,), device=DEVICE)) % N_CLASSES

            # Discriminator step
            real = torch.ones(batch_size, 1, device=DEVICE)
            fake = torch.zeros(batch_size, 1, device=DEVICE)

            # Real image, correct label
            pred_real = D(imgs, labels)
            loss_real = criterion_bce(pred_real, real)

            # Fake image, correct label
            pred_fake = D(x_fake, fake_labels)
            loss_fake = criterion_bce(pred_fake, fake)

            # Real image, wrong label
            pred_wrong = D(imgs, wrong_labels)
            loss_wrong = criterion_bce(pred_wrong, fake)

            # Gradient penalty
            gp = compute_gradient_penalty(D, imgs.data, x_fake.data, labels, DEVICE, lambda_gp=lambda_gp)

            # Total D loss
            d_loss = loss_real + loss_fake + loss_wrong + gp
            optimizer_D.zero_grad()
            d_loss.backward()
            optimizer_D.step()

            # Generator step
            z = torch.randn(batch_size, NOISE_DIM, device=DEVICE)
            gen_labels = torch.randint(0, N_CLASSES, (batch_size,), device=DEVICE)
            z_gen = G(z, gen_labels)
            x_gen = decoder(z_gen, gen_labels)
            pred = D(x_gen, gen_labels)
            g_loss = criterion_bce(pred, real)
            optimizer_G.zero_grad()
            g_loss.backward()
            optimizer_G.step()

        d_losses.append(d_loss.item())
        g_losses.append(g_loss.item())

    # Store results for this combination
    sweep_results.append({
        'lr_G': lr_G,
        'lr_D': lr_D,
        'lambda_gp': lambda_gp,
        'd_losses': d_losses,
        'g_losses': g_losses
    })
    print(f"Finished combination: lr_G={lr_G}, lr_D={lr_D}, LAMBDA_GP={lambda_gp} | Final D_loss: {d_losses[-1]:.4f} | Final G_loss: {g_losses[-1]:.4f}")


Testing combination: lr_G=0.0001, lr_D=0.0001, LAMBDA_GP=5
Finished combination: lr_G=0.0001, lr_D=0.0001, LAMBDA_GP=5 | Final D_loss: 1.3802 | Final G_loss: 3.0389
Testing combination: lr_G=0.0001, lr_D=0.0001, LAMBDA_GP=10
Finished combination: lr_G=0.0001, lr_D=0.0001, LAMBDA_GP=10 | Final D_loss: 1.5821 | Final G_loss: 2.0996
Testing combination: lr_G=0.0001, lr_D=0.0001, LAMBDA_GP=15
Finished combination: lr_G=0.0001, lr_D=0.0001, LAMBDA_GP=15 | Final D_loss: 1.6614 | Final G_loss: 1.8350
Testing combination: lr_G=0.0001, lr_D=0.0002, LAMBDA_GP=5
Finished combination: lr_G=0.0001, lr_D=0.0002, LAMBDA_GP=5 | Final D_loss: 1.2420 | Final G_loss: 3.4140
Testing combination: lr_G=0.0001, lr_D=0.0002, LAMBDA_GP=10
Finished combination: lr_G=0.0001, lr_D=0.0002, LAMBDA_GP=10 | Final D_loss: 1.3897 | Final G_loss: 2.7428
Testing combination: lr_G=0.0001, lr_D=0.0002, LAMBDA_GP=15
Finished combination: lr_G=0.0001, lr_D=0.0002, LAMBDA_GP=15 | Final D_loss: 1.4985 | Final G_loss: 2.1255
Te

Evaluate sweep results
Select best hyperparameters
Train with best hyperparameters
Finish task